# LOC Isolation Forest — MNR FFS

Multivariate anomaly detection on authorization rate features.  
Two-pass approach:
- **Pass 1 (Core)**: `initial_adr_rate`, `persistent_adr_rate`, `p2p_overturn_rate`, `appeal_overturn_rate`, `mcr_overturn_rate`, `md_review_rate` — 202501–202512
- **Pass 2 (Member Appeal)**: `member_appeal_rate`, `member_appeal_overturn_rate` — 202601+

Dimensions: `prov_tin`, `hospital_group`, `fin_market`  
Minimum volume: ≥30 cases per entity

In [ ]:
from sqlalchemy import create_engine
from snowflake.sqlalchemy import URL
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

engine = create_engine(URL(
    account = "UHG-UHGDWAAS",
    user = "KHANG.NGUYEN@UHC.COM",
    authenticator = "externalbrowser",
    role = "AZU_SDRP_VING_PRD_DEVELOPER_ROLE",
    warehouse = "VING_PRD_MNR_HCE_DATAINFRA_WH",
    database = "VING_PRD_TREND_DB",
    schema = "TMP_1M"
))

## Pass 1 — Core Aggregation (202501–202512)

In [ ]:
# pull all 3 dimensions in one query using union all
df_core = pd.read_sql("""
    select 'prov_tin' as _dimension
        , substr(a.fa_prov_id, 2, 9) as entity
        , count(*) as case_count
        , sum(a.initialfulladr_cases) as initial_adr_cnt
        , sum(a.persistentfulladr_cases) as persistent_adr_cnt
        , sum(a.p2p_full_ovtn) as p2p_ovrtn_cnt
        , sum(a.appeal_ovrtn_ind) as appeal_ovrtn_cnt
        , sum(a.mcr_ovtrn_ind) as mcr_ovrtn_cnt
        , sum(a.icm_md_reviewed_ind) as md_reviewed_cnt
    from tmp_1m.ec_avtar_25_26_3_od as a
    where fin_brand in ('M&R', 'C&S')
        and (
            (ip_type in ('Medical', 'Surgical', 'Transplant')
             and to_varchar(admit_dt_act, 'MM/dd/yyyy') is not null)
            or ip_type in ('LTAC', 'SNF', 'AIR')
        )
        and fin_brand = 'M&R'
        and loc_flag = 1
        and (
            (global_cap = 'NA' and sgr_source_name = 'COSMOS'
             and fin_product_level_3 != 'INSTITUTIONAL' and tfm_include_flag = 1)
            or (sgr_source_name = 'NICE' and nce_tadm_dec_risk_type in ('FFS', 'PHYSICIAN'))
        )
        and hce_admit_month between '202501' and '202512'
    group by 1, 2
    having count(*) >= 30

    union all

    select 'hospital_group' as _dimension
        , d.collection as entity
        , count(*) as case_count
        , sum(a.initialfulladr_cases) as initial_adr_cnt
        , sum(a.persistentfulladr_cases) as persistent_adr_cnt
        , sum(a.p2p_full_ovtn) as p2p_ovrtn_cnt
        , sum(a.appeal_ovrtn_ind) as appeal_ovrtn_cnt
        , sum(a.mcr_ovtrn_ind) as mcr_ovrtn_cnt
        , sum(a.icm_md_reviewed_ind) as md_reviewed_cnt
    from tmp_1m.ec_avtar_25_26_3_od as a
    left join tmp_1y.tin_collection as d
        on substr(a.fa_prov_id, 2, 9) = d.tin
    where fin_brand in ('M&R', 'C&S')
        and (
            (ip_type in ('Medical', 'Surgical', 'Transplant')
             and to_varchar(admit_dt_act, 'MM/dd/yyyy') is not null)
            or ip_type in ('LTAC', 'SNF', 'AIR')
        )
        and fin_brand = 'M&R'
        and loc_flag = 1
        and (
            (global_cap = 'NA' and sgr_source_name = 'COSMOS'
             and fin_product_level_3 != 'INSTITUTIONAL' and tfm_include_flag = 1)
            or (sgr_source_name = 'NICE' and nce_tadm_dec_risk_type in ('FFS', 'PHYSICIAN'))
        )
        and hce_admit_month between '202501' and '202512'
    group by 1, 2
    having count(*) >= 30

    union all

    select 'fin_market' as _dimension
        , a.fin_market as entity
        , count(*) as case_count
        , sum(a.initialfulladr_cases) as initial_adr_cnt
        , sum(a.persistentfulladr_cases) as persistent_adr_cnt
        , sum(a.p2p_full_ovtn) as p2p_ovrtn_cnt
        , sum(a.appeal_ovrtn_ind) as appeal_ovrtn_cnt
        , sum(a.mcr_ovtrn_ind) as mcr_ovrtn_cnt
        , sum(a.icm_md_reviewed_ind) as md_reviewed_cnt
    from tmp_1m.ec_avtar_25_26_3_od as a
    where fin_brand in ('M&R', 'C&S')
        and (
            (ip_type in ('Medical', 'Surgical', 'Transplant')
             and to_varchar(admit_dt_act, 'MM/dd/yyyy') is not null)
            or ip_type in ('LTAC', 'SNF', 'AIR')
        )
        and fin_brand = 'M&R'
        and loc_flag = 1
        and (
            (global_cap = 'NA' and sgr_source_name = 'COSMOS'
             and fin_product_level_3 != 'INSTITUTIONAL' and tfm_include_flag = 1)
            or (sgr_source_name = 'NICE' and nce_tadm_dec_risk_type in ('FFS', 'PHYSICIAN'))
        )
        and hce_admit_month between '202501' and '202512'
    group by 1, 2
    having count(*) >= 30
""", engine)

print(f"Core combined: {df_core.shape}")
df_core.groupby("_dimension").size()

In [ ]:
# compute rates
df_core["initial_adr_rate"] = df_core["initial_adr_cnt"] / df_core["case_count"]
df_core["persistent_adr_rate"] = df_core["persistent_adr_cnt"] / df_core["case_count"]
df_core["p2p_overturn_rate"] = df_core["p2p_ovrtn_cnt"] / df_core["initial_adr_cnt"].replace(0, np.nan)
df_core["appeal_overturn_rate"] = df_core["appeal_ovrtn_cnt"] / df_core["initial_adr_cnt"].replace(0, np.nan)
df_core["mcr_overturn_rate"] = df_core["mcr_ovrtn_cnt"] / df_core["initial_adr_cnt"].replace(0, np.nan)
df_core["md_review_rate"] = df_core["md_reviewed_cnt"] / df_core["case_count"]

# drop rows where overturn rates are NaN (zero initial ADRs)
df_core = df_core.dropna(subset = ["p2p_overturn_rate"])
print(f"After dropping NaN rates: {df_core.shape}")
df_core[["initial_adr_rate", "persistent_adr_rate", "p2p_overturn_rate",
         "appeal_overturn_rate", "mcr_overturn_rate", "md_review_rate"]].describe().round(4)

## EDA — Feature Distributions

In [ ]:
fig, axes = plt.subplots(3, 6, figsize = (20, 10))
for i, dim in enumerate(["prov_tin", "hospital_group", "fin_market"]):
    subset = df_core.query(f"_dimension == '{dim}'")
    for j, feat in enumerate(["initial_adr_rate", "persistent_adr_rate", "p2p_overturn_rate",
                              "appeal_overturn_rate", "mcr_overturn_rate", "md_review_rate"]):
        ax = axes[i, j]
        subset[feat].hist(bins = 30, ax = ax, edgecolor = "white")
        ax.set_title(f"{dim} — {feat}", fontsize = 9)
        ax.set_xlabel("")
plt.suptitle("Pass 1: Core Feature Distributions by Dimension", fontsize = 12)
plt.tight_layout()
plt.show()

## Core Isolation Forest

In [1]:
# fit IF per dimension
results_core = []

for dim in ["prov_tin", "hospital_group", "fin_market"]:
    subset = df_core.query(f"_dimension == '{dim}'").copy()

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(subset[["initial_adr_rate", "persistent_adr_rate",
                                           "p2p_overturn_rate", "appeal_overturn_rate",
                                           "mcr_overturn_rate", "md_review_rate"]])

    iso = IsolationForest(contamination = 0.05, n_estimators = 200, random_state = 42)
    iso.fit(X_scaled)

    subset["anomaly_score"] = iso.decision_function(X_scaled)
    subset["is_anomaly"] = (iso.predict(X_scaled) == -1).astype(int)

    # z-scores for interpretability
    subset[["z_initial_adr_rate", "z_persistent_adr_rate", "z_p2p_overturn_rate",
            "z_appeal_overturn_rate", "z_mcr_overturn_rate", "z_md_review_rate"]] = X_scaled

    # top reason: feature with highest absolute z-score + direction
    z_cols = ["z_initial_adr_rate", "z_persistent_adr_rate", "z_p2p_overturn_rate",
              "z_appeal_overturn_rate", "z_mcr_overturn_rate", "z_md_review_rate"]
    subset["top_feature"] = subset[z_cols].abs().idxmax(axis = 1).str.replace("z_", "")
    subset["top_z"] = subset[z_cols].apply(lambda row: row[row.abs().idxmax()], axis = 1)
    subset["top_reason"] = subset["top_feature"] + " (" + np.where(subset["top_z"] > 0, "high", "low") + ")"

    results_core.append(subset)
    n_flagged = subset["is_anomaly"].sum()
    print(f"{dim}: {len(subset)} entities, {n_flagged} flagged ({n_flagged / len(subset):.1%})")

df_core_results = pd.concat(results_core, ignore_index = True)
print(f"\nTotal core results: {df_core_results.shape}")

NameError: name 'df_core' is not defined

In [ ]:
# inspect flagged entities
display(
    df_core_results
    .query("is_anomaly == 1")
    .sort_values("anomaly_score")
    [["_dimension", "entity", "case_count", "initial_adr_rate",
      "persistent_adr_rate", "p2p_overturn_rate", "appeal_overturn_rate",
      "mcr_overturn_rate", "md_review_rate", "anomaly_score", "top_reason"]]
    .round(4)
    .head(20)
)

## Pass 2 — Member Appeal (202601+)

In [ ]:
# member appeal aggregation — all 3 dimensions
df_ma = pd.read_sql("""
    select 'prov_tin' as _dimension
        , substr(a.fa_prov_id, 2, 9) as entity
        , count(*) as case_count
        , sum(a.initialfulladr_cases) as initial_adr_cnt
        , sum(a.member_appeal_ind) as member_appeal_cnt
        , sum(a.member_appeal_ovtn_ind) as member_appeal_ovtn_cnt
    from tmp_1m.ec_avtar_25_26_3_od as a
    where fin_brand in ('M&R', 'C&S')
        and (
            (ip_type in ('Medical', 'Surgical', 'Transplant')
             and to_varchar(admit_dt_act, 'MM/dd/yyyy') is not null)
            or ip_type in ('LTAC', 'SNF', 'AIR')
        )
        and fin_brand = 'M&R'
        and loc_flag = 1
        and (
            (global_cap = 'NA' and sgr_source_name = 'COSMOS'
             and fin_product_level_3 != 'INSTITUTIONAL' and tfm_include_flag = 1)
            or (sgr_source_name = 'NICE' and nce_tadm_dec_risk_type in ('FFS', 'PHYSICIAN'))
        )
        and hce_admit_month >= '202601'
    group by 1, 2
    having count(*) >= 30

    union all

    select 'hospital_group' as _dimension
        , d.collection as entity
        , count(*) as case_count
        , sum(a.initialfulladr_cases) as initial_adr_cnt
        , sum(a.member_appeal_ind) as member_appeal_cnt
        , sum(a.member_appeal_ovtn_ind) as member_appeal_ovtn_cnt
    from tmp_1m.ec_avtar_25_26_3_od as a
    left join tmp_1y.tin_collection as d
        on substr(a.fa_prov_id, 2, 9) = d.tin
    where fin_brand in ('M&R', 'C&S')
        and (
            (ip_type in ('Medical', 'Surgical', 'Transplant')
             and to_varchar(admit_dt_act, 'MM/dd/yyyy') is not null)
            or ip_type in ('LTAC', 'SNF', 'AIR')
        )
        and fin_brand = 'M&R'
        and loc_flag = 1
        and (
            (global_cap = 'NA' and sgr_source_name = 'COSMOS'
             and fin_product_level_3 != 'INSTITUTIONAL' and tfm_include_flag = 1)
            or (sgr_source_name = 'NICE' and nce_tadm_dec_risk_type in ('FFS', 'PHYSICIAN'))
        )
        and hce_admit_month >= '202601'
    group by 1, 2
    having count(*) >= 30

    union all

    select 'fin_market' as _dimension
        , a.fin_market as entity
        , count(*) as case_count
        , sum(a.initialfulladr_cases) as initial_adr_cnt
        , sum(a.member_appeal_ind) as member_appeal_cnt
        , sum(a.member_appeal_ovtn_ind) as member_appeal_ovtn_cnt
    from tmp_1m.ec_avtar_25_26_3_od as a
    where fin_brand in ('M&R', 'C&S')
        and (
            (ip_type in ('Medical', 'Surgical', 'Transplant')
             and to_varchar(admit_dt_act, 'MM/dd/yyyy') is not null)
            or ip_type in ('LTAC', 'SNF', 'AIR')
        )
        and fin_brand = 'M&R'
        and loc_flag = 1
        and (
            (global_cap = 'NA' and sgr_source_name = 'COSMOS'
             and fin_product_level_3 != 'INSTITUTIONAL' and tfm_include_flag = 1)
            or (sgr_source_name = 'NICE' and nce_tadm_dec_risk_type in ('FFS', 'PHYSICIAN'))
        )
        and hce_admit_month >= '202601'
    group by 1, 2
    having count(*) >= 30
""", engine)

engine.dispose()
print(f"Member appeal combined: {df_ma.shape}")
df_ma.groupby("_dimension").size()

In [ ]:
# compute member appeal rates
df_ma["member_appeal_rate"] = df_ma["member_appeal_cnt"] / df_ma["initial_adr_cnt"].replace(0, np.nan)
df_ma["member_appeal_overturn_rate"] = df_ma["member_appeal_ovtn_cnt"] / df_ma["initial_adr_cnt"].replace(0, np.nan)

df_ma = df_ma.dropna(subset = ["member_appeal_rate", "member_appeal_overturn_rate"])
print(f"After dropping NaN: {df_ma.shape}")
df_ma[["member_appeal_rate", "member_appeal_overturn_rate"]].describe().round(4)

In [ ]:
# member appeal IF per dimension
results_ma = []

for dim in ["prov_tin", "hospital_group", "fin_market"]:
    subset = df_ma.query(f"_dimension == '{dim}'").copy()

    if len(subset) < 10:
        print(f"{dim}: only {len(subset)} entities — skipping IF")
        continue

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(subset[["member_appeal_rate", "member_appeal_overturn_rate"]])

    iso = IsolationForest(contamination = 0.05, n_estimators = 200, random_state = 42)
    iso.fit(X_scaled)

    subset["anomaly_score"] = iso.decision_function(X_scaled)
    subset["is_anomaly"] = (iso.predict(X_scaled) == -1).astype(int)

    subset[["z_member_appeal_rate", "z_member_appeal_overturn_rate"]] = X_scaled

    subset["top_feature"] = (subset[["z_member_appeal_rate", "z_member_appeal_overturn_rate"]]
                             .abs().idxmax(axis = 1).str.replace("z_", ""))
    subset["top_z"] = subset[["z_member_appeal_rate", "z_member_appeal_overturn_rate"]].apply(
        lambda row: row[row.abs().idxmax()], axis = 1)
    subset["top_reason"] = subset["top_feature"] + " (" + np.where(subset["top_z"] > 0, "high", "low") + ")"

    results_ma.append(subset)
    n_flagged = subset["is_anomaly"].sum()
    print(f"{dim}: {len(subset)} entities, {n_flagged} flagged ({n_flagged / len(subset):.1%})")

df_ma_results = pd.concat(results_ma, ignore_index = True) if results_ma else pd.DataFrame()
print(f"\nTotal MA results: {df_ma_results.shape}")

In [ ]:
# inspect flagged MA entities
if not df_ma_results.empty:
    display(
        df_ma_results
        .query("is_anomaly == 1")
        .sort_values("anomaly_score")
        [["_dimension", "entity", "case_count", "member_appeal_rate",
          "member_appeal_overturn_rate", "anomaly_score", "top_reason"]]
        .round(4)
        .head(20)
    )

## Combine + Interpret

In [ ]:
# combine core + MA flags
core_flags = (
    df_core_results
    .query("is_anomaly == 1")
    [["_dimension", "entity", "case_count", "initial_adr_rate",
      "persistent_adr_rate", "p2p_overturn_rate", "appeal_overturn_rate",
      "mcr_overturn_rate", "md_review_rate", "anomaly_score", "top_reason"]]
    .assign(flag_source = "core")
)

if not df_ma_results.empty:
    ma_flags = (
        df_ma_results
        .query("is_anomaly == 1")
        [["_dimension", "entity", "case_count", "member_appeal_rate",
          "member_appeal_overturn_rate", "anomaly_score", "top_reason"]]
        .assign(flag_source = "member_appeal")
    )
    combined = pd.concat([core_flags, ma_flags], ignore_index = True)
else:
    combined = core_flags.copy()

combined = combined.sort_values(["_dimension", "anomaly_score"])
print(f"Total combined flags: {len(combined)}")
print()
combined.groupby(["_dimension", "flag_source"]).size()

In [ ]:
# top anomalies per dimension (core)
for dim in ["prov_tin", "hospital_group", "fin_market"]:
    print(f"\n{'=' * 60}")
    print(f"Top 10 anomalies — {dim} (Core IF)")
    print(f"{'=' * 60}")
    display(
        df_core_results
        .query(f"_dimension == '{dim}' and is_anomaly == 1")
        .sort_values("anomaly_score")
        [["entity", "case_count", "initial_adr_rate", "persistent_adr_rate",
          "p2p_overturn_rate", "appeal_overturn_rate", "mcr_overturn_rate",
          "md_review_rate", "anomaly_score", "top_reason"]]
        .head(10)
        .round(4)
    )

## Export

In [ ]:
df_core_results.to_csv(r"C:\Users\knguy139\Documents\Projects\Data\Output\loc_if_core_mnr.csv", index = False)

if not df_ma_results.empty:
    df_ma_results.to_csv(r"C:\Users\knguy139\Documents\Projects\Data\Output\loc_if_member_appeal_mnr.csv", index = False)

combined.to_csv(r"C:\Users\knguy139\Documents\Projects\Data\Output\loc_if_combined_mnr.csv", index = False)

print(f"Exported:")
print(f"  loc_if_core_mnr.csv ({len(df_core_results)} rows)")
if not df_ma_results.empty:
    print(f"  loc_if_member_appeal_mnr.csv ({len(df_ma_results)} rows)")
print(f"  loc_if_combined_mnr.csv ({len(combined)} rows)")